In [ ]:
# Core
import pandas as pd
import numpy as np
import time

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

from plot3HeatMaps import plot_three_heatmaps
from static_training import train_static
from progressive_training import train_progressive
from mistake_driven_training import train_mistake_driven


In [ ]:
df = pd.read_csv("data/electricity_market_dataset.csv")
# Convert timestamp
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Sort by time (VERY IMPORTANT)
df = df.sort_values('Timestamp').reset_index(drop=True)

df.head()
print("Shape:", df.shape)


Format the data

In [ ]:
target = "Electricity_Price_Forecast"
df_model = df.copy()

# Encode categorical columns
df_model["Regulatory_Policies"] = df_model["Regulatory_Policies"].map({"Low": 0, "Medium": 1, "High": 2})
df_model["Energy_Access_Data"] = df_model["Energy_Access_Data"].map({"Rural": 0, "Urban": 1})
df_model["Project_Risk_Analysis"] = df_model["Project_Risk_Analysis"].map({"Low": 0, "Medium": 1, "High": 2})


# Drop target and timestamps
X = df_model.drop(columns=[target, "Timestamp"])
y = df_model[target]

In [ ]:
# Plot Function
def makeRSMEPlot(df_results, vmin=.8, vmax=25, title="RMSE Heatmap"):
    plt.figure(figsize=(8, 6))
    sns.heatmap(df_results.astype(float), annot=True, fmt=".2f", cmap="coolwarm",
                vmin=vmin, vmax=vmax)
    plt.title(title)
    plt.xlabel("Look-Ahead")
    plt.ylabel("Look-Back")
    plt.show()
def makeTimePlot(df_time, vmin=0, vmax=9, title="Computation Time (s)"):
    plt.figure(figsize=(8, 6))
    sns.heatmap(df_time.astype(float), annot=True, fmt=".2f", cmap="coolwarm",
                vmin=vmin, vmax=vmax)
    plt.title(title)
    plt.xlabel("Look-Ahead")
    plt.ylabel("Look-Back")
    plt.show()

In [ ]:
# Training window sizes (1, 6, 12, 18, 24 months)
look_back = [720, 4320, 8640,  12960, 17280]  
tree_depth = 5
# Retrain 3 Months
retrain_step = 24*90
# Look ahead 1 Month 
look_ahead = [720, 4320, 8640,  12960, 17280]
# retrain if RMSE > 2.5
threshold = 1

In [ ]:

# Dictionaries to store RMSE and computation time
results_static = {}
time_static = {}

for lb in look_back:
    results_static[lb] = {}
    time_static[lb] = {}
    for la in look_ahead:
        start_time = time.time()
        avg_rmse = train_static(X, y, look_back=lb, look_ahead=la, depthOfTree=tree_depth)
        elapsed_time = time.time() - start_time

        # Store
        results_static[lb][la] = avg_rmse
        time_static[lb][la] = elapsed_time

        print(f"Look-back {lb}, Look-ahead {la} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")

We will use the function to look ahead by 1 month, retraining every 3 months. Using the same training lengths and decision tree sizes

In [ ]:
results_progressive = {}
time_progressive = {}

for lb in look_back:
    results_progressive[lb] = {}
    time_progressive[lb] = {}
    for la in look_ahead:
        start_time = time.time()
        avg_rmse = train_progressive(X, y, look_back=lb, look_ahead=la,
                                        retrain_step=retrain_step, depthOfTree=tree_depth)
        elapsed_time = time.time() - start_time
        time_progressive[lb][la] = elapsed_time
        results_progressive[lb][la] = avg_rmse
        print(f"Look-back {lb}, Look-ahead {la} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")


In [ ]:
results_mistake = {}
time_mistake = {}

for lb in look_back:
    results_mistake[lb] = {}
    time_mistake[lb] = {}
    for la in look_ahead:
        start_time = time.time()
        avg_rmse = train_mistake_driven(X, y, look_back=lb, look_ahead=la,
                                           depthOfTree=tree_depth, threshold=threshold)
        elapsed_time = time.time() - start_time
        
        results_mistake[lb][la] = avg_rmse
        time_mistake[lb][la] = elapsed_time
        
        print(f"Look-back {lb}, Look-ahead {la} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")

In [ ]:
df_static_rmse = pd.DataFrame(results_static).T
df_static_time = pd.DataFrame(time_static).T

df_progressive_rmse = pd.DataFrame(results_progressive).T
df_progressive_time = pd.DataFrame(time_progressive).T

df_mistake_rmse = pd.DataFrame(results_mistake).T
df_mistake_time = pd.DataFrame(time_mistake).T

In [ ]:
plot_three_heatmaps(
    dfs=[df_static_rmse, df_progressive_rmse, df_mistake_rmse],
    titles=["Static RMSE", "Progressive RMSE", "Mistake-Driven RMSE"],
    vmin=min(df_static_rmse.min().min(), df_progressive_rmse.min().min(), df_mistake_rmse.min().min()),
    vmax=max(df_static_rmse.max().max(), df_progressive_rmse.max().max(), df_mistake_rmse.max().max())
)


plot_three_heatmaps(
    dfs=[df_static_time, df_progressive_time, df_mistake_time],
    titles=["Static Time (s)", "Progressive Time (s)", "Mistake-Driven Time (s)"],
    vmin=min(df_static_time.min().min(), df_progressive_time.min().min(), df_mistake_time.min().min()),
    vmax=max(df_static_time.max().max(), df_progressive_time.max().max(), df_mistake_time.max().max())
)


# Conclusion
For the electrical project the time savings seem to have an inverse relationship with the RSME value that comes back. If the retrain is set to a competitive level with progressive window it is possible to get RSME within striking distance struggles to keep things going